# 📊 03 — Evaluación del Modelo
**Proyecto 01: Reciclaje Inteligente — Clasificación de Desechos con Visión Artificial**  
Grupo #4 · Curso 045 — Inteligencia Artificial · UMG 2026  
Responsable: Javier Rivera / Paolo Marroquín

---
## Objetivos de este notebook
1. Cargar el modelo entrenado `models/modelo_reciclaje.pth`
2. Calcular métricas globales en el conjunto de **test**: Accuracy, F1-macro, Top-2 Accuracy
3. Generar el **reporte de clasificación** por clase (precision, recall, F1 por clase)
4. Visualizar la **matriz de confusión** (heatmap)
5. Analizar los **errores más frecuentes** del modelo con ejemplos visuales
6. Presentar el resumen ejecutivo con conclusiones

---
> ⚠️ **Pre-requisito:** Haber ejecutado `notebooks/02_train.ipynb` para generar `models/modelo_reciclaje.pth`

## 0. Configuración e importaciones

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

sys.path.insert(0, os.path.join('..', 'src'))
from model import load_model, CLASSES, COLOR_MAP
from preprocess import get_val_transforms

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── Rutas ─────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../data/processed/test')
MODEL_PATH = Path('../models/modelo_reciclaje.pth')
DOCS_DIR   = Path('../docs')
DOCS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('✅ Librerías importadas correctamente')
print(f'  Dispositivo: {device}')
print(f'  Modelo:      {MODEL_PATH.resolve()}')
print(f'  Test set:    {DATA_DIR.resolve()}')

# Verificar que el modelo existe
if not MODEL_PATH.exists():
    print(f'\n❌ Modelo no encontrado: {MODEL_PATH}')
    print('   Ejecuta primero: notebooks/02_train.ipynb')
else:
    print(f'\n✅ Modelo encontrado ({MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MB)')

## 1. Carga del modelo y dataset de prueba

In [ ]:
# ── Cargar modelo ─────────────────────────────────────────────────────────
model = load_model(str(MODEL_PATH))
model = model.to(device)
model.eval()
print('✅ Modelo cargado en modo evaluación')

# ── Cargar dataset de test ────────────────────────────────────────────────
test_ds = ImageFolder(str(DATA_DIR), transform=get_val_transforms())
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

print(f'\n📦 Dataset de test cargado:')
print(f'   Imágenes totales: {len(test_ds)}')
print(f'   Clases ({len(test_ds.classes)}): {test_ds.classes}')

# Verificar que las clases del test coinciden con las del modelo
mismatches = [c for c in test_ds.classes if c not in CLASSES]
if mismatches:
    print(f'\n⚠️  Clases en test que NO están en el modelo: {mismatches}')
else:
    print(f'\n✅ Clases del test alineadas con el modelo')

## 2. Inferencia en el conjunto de test

In [ ]:
print('🔍 Realizando inferencia en el conjunto de test...')

all_preds      = []
all_targets    = []
all_probs      = []
all_img_paths  = [p for p, _ in test_ds.imgs]

with torch.no_grad():
    for batch_idx, (imgs, labels) in enumerate(test_loader):
        imgs = imgs.to(device)
        logits = model(imgs)
        probs  = torch.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_targets.extend(labels.tolist())
        all_probs.extend(probs.cpu().tolist())

        if (batch_idx + 1) % 5 == 0 or batch_idx == len(test_loader) - 1:
            print(f'   Batch {batch_idx + 1}/{len(test_loader)} procesado...')

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)
all_probs   = np.array(all_probs)

print(f'\n✅ Inferencia completada — {len(all_preds)} predicciones generadas')

## 3. Métricas globales

In [ ]:
# ── Métricas principales ──────────────────────────────────────────────────
acc_global = accuracy_score(all_targets, all_preds)
f1_macro   = f1_score(all_targets, all_preds, average='macro',  zero_division=0)
f1_weighted= f1_score(all_targets, all_preds, average='weighted', zero_division=0)

# Top-2 Accuracy
top2_correct = 0
for i, target in enumerate(all_targets):
    top2_idx = np.argsort(all_probs[i])[-2:]  # índices de los 2 más probables
    if target in top2_idx:
        top2_correct += 1
top2_acc = top2_correct / len(all_targets)

# Confianza promedio en predicciones correctas vs incorrectas
conf_correctas   = np.mean([all_probs[i][all_preds[i]] for i in range(len(all_preds)) if all_preds[i] == all_targets[i]])
conf_incorrectas = np.mean([all_probs[i][all_preds[i]] for i in range(len(all_preds)) if all_preds[i] != all_targets[i]])

print('=' * 55)
print('  MÉTRICAS GLOBALES — Conjunto de Test')
print('  Reciclaje Inteligente · Grupo #4 · UMG 2026')
print('=' * 55)
print(f'\n  Accuracy global:        {acc_global*100:>7.2f}%')
print(f'  F1-macro:               {f1_macro*100:>7.2f}%')
print(f'  F1-weighted:            {f1_weighted*100:>7.2f}%')
print(f'  Top-2 Accuracy:         {top2_acc*100:>7.2f}%')
print(f'\n  Confianza media (✅):   {conf_correctas*100:>7.2f}%')
print(f'  Confianza media (❌):   {conf_incorrectas*100:>7.2f}%')
print(f'\n  Imágenes en test:       {len(all_preds):>7}')
print(f'  Correctas:              {int(acc_global * len(all_preds)):>7}')
print(f'  Incorrectas:            {len(all_preds) - int(acc_global * len(all_preds)):>7}')

# Evaluación cualitativa
if acc_global >= 0.85:
    nivel = '🏆 Excelente'
elif acc_global >= 0.70:
    nivel = '✅ Bueno'
elif acc_global >= 0.55:
    nivel = '⚠️  Aceptable'
else:
    nivel = '❌ Necesita mejora'
print(f'\n  Evaluación: {nivel}')

## 4. Reporte de clasificación por clase

In [ ]:
# ── Classification Report textual ─────────────────────────────────────────
labels_nombres = [test_ds.classes[i] for i in sorted(set(all_targets))]
reporte_texto  = classification_report(
    all_targets, all_preds,
    target_names=test_ds.classes,
    zero_division=0
)
print('📋 REPORTE DE CLASIFICACIÓN POR CLASE')
print('=' * 65)
print(reporte_texto)

# ── Visualización en barras horizontales ──────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1_por_clase, soporte = precision_recall_fscore_support(
    all_targets, all_preds, labels=list(range(len(test_ds.classes))),
    average=None, zero_division=0
)

clases_plot = test_ds.classes
x = np.arange(len(clases_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
bars_p = ax.barh(x + width, precision * 100, height=width, label='Precisión',  color='#2E86C1', alpha=0.85)
bars_r = ax.barh(x,         recall    * 100, height=width, label='Recall',     color='#27AE60', alpha=0.85)
bars_f = ax.barh(x - width, f1_por_clase * 100, height=width, label='F1-score', color='#E74C3C', alpha=0.85)

ax.set_yticks(x)
ax.set_yticklabels(clases_plot, fontsize=10)
ax.set_xlabel('Porcentaje (%)', fontsize=11)
ax.set_title('Métricas por clase — Conjunto de Test\nReciclaje Inteligente · Grupo #4 · UMG 2026',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 115)
ax.legend(loc='lower right', fontsize=10)
ax.axvline(70, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Umbral 70%')
ax.grid(axis='x', alpha=0.3)

# Añadir valores
for b_list, vals in [(bars_p, precision), (bars_r, recall), (bars_f, f1_por_clase)]:
    for bar, val in zip(b_list, vals):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f'{val*100:.1f}', va='center', fontsize=7.5, color='#2C3E50')

plt.tight_layout()
plt.savefig(str(DOCS_DIR / 'eval_metricas_por_clase.png'), bbox_inches='tight', dpi=150)
plt.show()
print('💾 Guardado en docs/eval_metricas_por_clase.png')

## 5. Matriz de confusión

In [ ]:
cm     = confusion_matrix(all_targets, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ── Matriz absoluta ───────────────────────────────────────────────────────
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=test_ds.classes, yticklabels=test_ds.classes,
    linewidths=0.5, linecolor='white',
    ax=axes[0], cbar_kws={'label': 'Cantidad'},
    annot_kws={'size': 9}
)
axes[0].set_title('Matriz de Confusión — Conteo absoluto', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicción', fontsize=11)
axes[0].set_ylabel('Clase Real', fontsize=11)
axes[0].tick_params(axis='x', rotation=45, labelsize=9)
axes[0].tick_params(axis='y', rotation=0,  labelsize=9)

# ── Matriz porcentual (normalizada por fila) ───────────────────────────────
sns.heatmap(
    cm_pct, annot=True, fmt='.1f', cmap='YlOrRd',
    xticklabels=test_ds.classes, yticklabels=test_ds.classes,
    linewidths=0.5, linecolor='white',
    ax=axes[1], cbar_kws={'label': '% por clase real'},
    vmin=0, vmax=100,
    annot_kws={'size': 9}
)
axes[1].set_title('Matriz de Confusión — Porcentaje por clase real', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicción', fontsize=11)
axes[1].set_ylabel('Clase Real', fontsize=11)
axes[1].tick_params(axis='x', rotation=45, labelsize=9)
axes[1].tick_params(axis='y', rotation=0,  labelsize=9)

plt.suptitle(
    'Matriz de Confusión — MobileNetV2 · Reciclaje Inteligente · Grupo #4 · UMG 2026',
    fontsize=11, y=1.01, color='gray'
)
plt.tight_layout()
plt.savefig(str(DOCS_DIR / 'eval_matriz_confusion.png'), bbox_inches='tight', dpi=150)
plt.show()
print('💾 Guardado en docs/eval_matriz_confusion.png')

# ── Pares más confundidos ─────────────────────────────────────────────────
print('\n🔍 Top-5 pares de clases más confundidas (fuera de la diagonal):')
errores = []
for i in range(len(test_ds.classes)):
    for j in range(len(test_ds.classes)):
        if i != j and cm[i, j] > 0:
            errores.append((cm[i, j], test_ds.classes[i], test_ds.classes[j]))
errores.sort(reverse=True)
for n, real, pred in errores[:5]:
    print(f'   {real:<28} → predijo {pred:<28}  ({n} veces)')

## 6. Análisis de errores — Ejemplos visuales

In [ ]:
# ── Encontrar predicciones incorrectas ─────────────────────────────────────
indices_erroneos = np.where(all_preds != all_targets)[0]
print(f'❌ Total de predicciones incorrectas: {len(indices_erroneos)} de {len(all_preds)} ({len(indices_erroneos)/len(all_preds)*100:.1f}%)')

if len(indices_erroneos) == 0:
    print('\n🏆 ¡El modelo clasificó TODAS las imágenes correctamente!')
else:
    # Ordenar por confianza para mostrar los más "seguros pero equivocados"
    indices_erroneos_conf = sorted(
        indices_erroneos,
        key=lambda i: all_probs[i][all_preds[i]],
        reverse=True
    )[:6]

    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
    IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

    n_mostrar = min(6, len(indices_erroneos_conf))
    fig, axes = plt.subplots(2, 3, figsize=(14, 9)) if n_mostrar > 3 else plt.subplots(1, n_mostrar, figsize=(4 * n_mostrar, 4))
    axes = np.array(axes).flatten()

    for plot_idx, err_idx in enumerate(indices_erroneos_conf[:n_mostrar]):
        # Cargar imagen original
        img_path = test_ds.imgs[err_idx][0]
        img = Image.open(img_path).convert('RGB').resize((224, 224))

        clase_real  = test_ds.classes[all_targets[err_idx]]
        clase_pred  = test_ds.classes[all_preds[err_idx]]
        confianza   = all_probs[err_idx][all_preds[err_idx]] * 100

        emoji_real = COLOR_MAP.get(clase_real, ('', '', '❓', ''))[2]
        emoji_pred = COLOR_MAP.get(clase_pred, ('', '', '❓', ''))[2]

        ax = axes[plot_idx]
        ax.imshow(img)
        ax.set_title(
            f'Real: {emoji_real} {clase_real}\n'
            f'Predijo: {emoji_pred} {clase_pred} ({confianza:.1f}%)',
            fontsize=9, color='#C0392B', fontweight='bold'
        )
        ax.axis('off')
        # Bordes rojo = error
        for spine in ax.spines.values():
            spine.set_edgecolor('#E74C3C')
            spine.set_linewidth(2.5)
            spine.set_visible(True)

    # Ocultar ejes sobrantes
    for ax in axes[n_mostrar:]:
        ax.axis('off')

    plt.suptitle(
        f'Análisis de Errores — {n_mostrar} predicciones incorrectas con mayor confianza\n'
        f'Reciclaje Inteligente · Grupo #4 · UMG 2026',
        fontsize=11, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig(str(DOCS_DIR / 'eval_analisis_errores.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print('💾 Guardado en docs/eval_analisis_errores.png')

## 7. Resumen ejecutivo y conclusiones

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec_pc, rec_pc, f1_pc, soporte_pc = precision_recall_fscore_support(
    all_targets, all_preds,
    labels=list(range(len(test_ds.classes))),
    average=None, zero_division=0
)

print('=' * 70)
print('  RESUMEN EJECUTIVO — Evaluación del Modelo')
print('  Reciclaje Inteligente · Clasificación de Desechos · Grupo #4 · UMG 2026')
print('=' * 70)

print(f'\n  MÉTRICAS GLOBALES')
print(f'  ┌─────────────────────────┬────────────┐')
print(f'  │ Métrica                 │   Valor    │')
print(f'  ├─────────────────────────┼────────────┤')
print(f'  │ Accuracy global         │ {acc_global*100:>8.2f}%  │')
print(f'  │ F1-macro                │ {f1_macro*100:>8.2f}%  │')
print(f'  │ F1-weighted             │ {f1_weighted*100:>8.2f}%  │')
print(f'  │ Top-2 Accuracy          │ {top2_acc*100:>8.2f}%  │')
print(f'  │ Imágenes en test        │ {len(all_preds):>10} │')
print(f'  └─────────────────────────┴────────────┘')

print(f'\n  MÉTRICAS POR CLASE')
print(f'  {"Clase":<28} {"Prec":>6} {"Recall":>7} {"F1":>6} {"n":>5} {"Estado"}')
print(f'  {"-"*65}')
for i, clase in enumerate(test_ds.classes):
    emoji = COLOR_MAP.get(clase, ('', '', '  ', ''))[2]
    est = ('✅' if f1_pc[i] >= 0.70 else ('⚠️' if f1_pc[i] >= 0.50 else '❌'))
    print(f'  {emoji} {clase:<26} {prec_pc[i]*100:>5.1f}% {rec_pc[i]*100:>6.1f}% {f1_pc[i]*100:>5.1f}% {int(soporte_pc[i]):>5}  {est}')

# Clases con mejor y peor desempeño
mejor_clase = test_ds.classes[np.argmax(f1_pc)]
peor_clase  = test_ds.classes[np.argmin(f1_pc)]

print(f'\n  🏆 Mejor clase:  {mejor_clase} (F1={np.max(f1_pc)*100:.1f}%)')
print(f'  ⚠️  Peor clase:   {peor_clase} (F1={np.min(f1_pc)*100:.1f}%)')

print(f'\n  LIMITACIONES Y CONCLUSIONES')
print(f'  • Este sistema es una herramienta educativa de apoyo, NO una autoridad.')
print(f'  • El modelo puede fallar con imágenes en condiciones inusuales (luz, ángulo).')
print(f'  • Para clases con F1 < 70%, se recomienda recolectar más imágenes propias.')
print(f'  • El nivel de confianza se muestra al usuario para facilitar decisiones informadas.')

print(f'\n  ARCHIVOS GENERADOS')
archivos = [
    'docs/eval_metricas_por_clase.png',
    'docs/eval_matriz_confusion.png',
    'docs/eval_analisis_errores.png',
]
for f in archivos:
    ruta = Path('..') / f
    existe = '✅' if ruta.exists() else '⏳'
    print(f'  {existe} {f}')

print(f'\n✅ Evaluación completada. Resultados listos para el informe final.')